In [4]:
import numpy as np

# Historical Liver Donor Dataset
# Features: [Age, BMI, Smoking Habit (0=No, 1=Yes), General Health Score (1-5)]
X_train = np.array([
    [28, 22.5, 0, 5],
    [55, 34.0, 1, 2],
    [35, 24.0, 0, 4],
    [45, 31.5, 1, 2],
    [24, 21.0, 0, 5],
    [58, 29.0, 0, 3],
    [40, 36.0, 1, 1],
    [30, 23.5, 0, 4],
])
y_train = np.array([1, 0, 1, 0, 1, 1, 0, 1])

class Scaler:
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1e-8

    def transform(self, X):
        return (X - self.mean) / self.std

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

class KNN:

  def __init__(self, k=3):
    self.k = k
    self.scaler = Scaler()

  def fit(self, X, y):
    self.X_train = self.scaler.fit_transform(X)
    self.y_train = y


  def _euclidean_distance(self, x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

  def predict(self, x_new):
    distances = []
    x_new = self.scaler.transform(x_new)
    for x_train in self.X_train:
      dist = self._euclidean_distance(x_new, x_train)
      distances.append(dist)

    k_indices = np.argsort(distances)[: self.k]
    k_nearest_labels = self.y_train[k_indices]

    unique_labels, counts = np.unique(k_nearest_labels, return_counts=True)
    return unique_labels[np.argmax(counts)]


knn = KNN(k=3)
knn.fit(X_train, y_train)
print('System Initialized: Historical data loaded & KNN model ready.')

System Initialized: Historical data loaded & KNN model ready.


In [9]:
def evaluate_liver_donor_rules(
    age, bmi, smoking, has_infection, has_chronic_disease
):
  """Applies mandatory medical exclusion criteria for live liver donation."""

  if age < 18 or age > 60:
    return (
        False,
        f'Rejected by Rules: Age {age} is outside the safe range (18-60 years).'
        ' Live donors generally need to be adults under 60.',
    )

  if bmi < 18.5 or bmi > 32.0:
    return (
        False,
        f'Rejected by Rules: BMI {bmi:.1f} is outside the acceptable range'
        ' (18.5 - 32.0).',
    )

  if has_infection:
    return (
        False,
        'Rejected by Rules: Active infection detected. Cannot proceed while'
        ' fighting an infection.',
    )

  if has_chronic_disease:
    return (
        False,
        'Rejected by Rules: History of chronic disease disqualifies live'
        ' organ donation.',
    )

  return True, 'Passed: All hard medical screening filters cleared.'

In [11]:
print('=== LIVE LIVER DONOR ELIGIBILITY QUESTIONNAIRE ===\n')

try:
  age = int(input('Enter your age (years): '))
  weight_kg = float(input('Enter your weight (kg): '))
  height_m = float(input('Enter your height (meters, e.g., 1.75): '))

  smoking_input = input('Do you smoke regularly? (yes/no): ').strip().lower()
  smoking = 1 if smoking_input == 'yes' else 0

  infection_input = (
      input('Do you have any active infections currently? (yes/no): ')
      .strip()
      .lower()
  )
  has_infection = True if infection_input == 'yes' else False

  chronic_input = (
      input('Do you have chronic diseases (diabetes, hepatitis)? (yes/no): ')
      .strip()
      .lower()
  )
  has_chronic_disease = True if chronic_input == 'yes' else False

  health_score = int(
      input('Rate your overall self-reported health from 1 (poor) to 5 (excellent): ')
  )

  bmi = weight_kg / (height_m**2)
  print(f'\n--- Calculated BMI: {bmi:.1f} ---')

  passed_rules, message = evaluate_liver_donor_rules(
      age, bmi, smoking, has_infection, has_chronic_disease
  )
  print(f'If-Else Block Status: {message}')

  if passed_rules:
    user_features = np.array([age, bmi, smoking, health_score])

    prediction = knn.predict(user_features)

    print('\n--- ML MODEL BLOCK (KNN) ---')
    if prediction == 1:
      print(
          'Result: 🟢 ELIGIBLE DONOR PROFILE. Based on historical data similarity,'
          ' your profile matches successful live liver donors. (Note: Clinical'
          ' lab tests are still required).'
      )
    else:
      print(
          'Result: 🔴 HIGH RISK PROFILE. While you passed initial safety rules,'
          ' historical patterns suggest potential elevated risks for this type'
          ' of donation profile.'
      )
  else:
    print(
        '\nProcess Terminated: Match discarded at the rule-based medical'
        ' screening stage.'
    )

except ValueError:
  print(
      '\nInvalid input format. Please restart the cell and enter numbers where'
      ' requested.'
  )

=== LIVE LIVER DONOR ELIGIBILITY QUESTIONNAIRE ===

Enter your age (years): 70
Enter your weight (kg): 55
Enter your height (meters, e.g., 1.75): 1.20
Do you smoke regularly? (yes/no): df
Do you have any active infections currently? (yes/no): df
Do you have chronic diseases (diabetes, hepatitis)? (yes/no): fds
Rate your overall self-reported health from 1 (poor) to 5 (excellent): 4

--- Calculated BMI: 38.2 ---
If-Else Block Status: Rejected by Rules: Age 70 is outside the safe range (18-60 years). Live donors generally need to be adults under 60.

Process Terminated: Match discarded at the rule-based medical screening stage.
